In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e9/train.csv
/kaggle/input/competitions/playground-series-s6e9/test.csv


In [2]:
# df_train= pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/train.csv')
# df_test= pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/test.csv')

In [3]:
from __future__ import annotations

from typing import Iterable, Optional, Sequence, Union

class CSVDataset:
    """Wraps a pandas DataFrame loaded from CSV and adds EDA helpers.

    The original data is kept in ``self._original`` so you can always
    call ``reset()`` to undo any cleaning steps.
    """

    def __init__(
        self,
        filepath: Optional[str] = None,
        dataframe: Optional[pd.DataFrame] = None,
        **read_csv_kwargs,
    ):
        """
        Parameters
        ----------
        filepath : str, optional
            Path to a CSV file. Ignored if ``dataframe`` is given.
        dataframe : pd.DataFrame, optional
            Use an existing DataFrame instead of reading from disk.
        **read_csv_kwargs
            Passed straight through to ``pandas.read_csv``
            (e.g. sep=';', na_values=['NA', '?'], encoding='latin-1').
        """
        if dataframe is not None:
            self.df = dataframe.copy()
            self.filepath = None
        elif filepath is not None:
            self.filepath = filepath
            self.df = pd.read_csv(filepath, **read_csv_kwargs)
        else:
            raise ValueError("Provide either `filepath` or `dataframe`.")

        # Keep a pristine copy so cleaning is reversible.
        self._original = self.df.copy()

    # ------------------------------------------------------------------ #
    # Basic dunder / utility
    # ------------------------------------------------------------------ #
    def __repr__(self) -> str:
        rows, cols = self.df.shape
        return f"<CSVDataset rows={rows} cols={cols} source={self.filepath!r}>"

    def __len__(self) -> int:
        return len(self.df)

    def reset(self) -> "CSVDataset":
        """Restore the DataFrame to its state right after loading."""
        self.df = self._original.copy()
        return self

    def head(self, n: int = 5) -> pd.DataFrame:
        return self.df.head(n)

    # ------------------------------------------------------------------ #
    # Column type helpers
    # ------------------------------------------------------------------ #
    def numeric_columns(self) -> list[str]:
        """Columns that pandas treats as numeric."""
        return self.df.select_dtypes(include=[np.number]).columns.tolist()

    def categorical_columns(self, unique_threshold: Optional[int] = None) -> list[str]:
        """Columns that look categorical.

        By default this returns object / string / category dtype columns.
        If ``unique_threshold`` is given, numeric columns with fewer than
        that many distinct values are also flagged (e.g. a 0/1 flag column).
        """
        cat = self.df.select_dtypes(
            include=["object", "string", "category"]
        ).columns.tolist()

        if unique_threshold is not None:
            for col in self.numeric_columns():
                if self.df[col].nunique(dropna=True) < unique_threshold:
                    cat.append(col)
        return cat

    # ------------------------------------------------------------------ #
    # MISSING / NaN VALUE HANDLING
    # ------------------------------------------------------------------ #
    def missing_report(self) -> pd.DataFrame:
        """Per-column count and percentage of missing values, worst first."""
        count = self.df.isna().sum()
        pct = (count / len(self.df) * 100).round(2)
        report = (
            pd.DataFrame({"missing_count": count, "missing_pct": pct})
            .sort_values("missing_count", ascending=False)
        )
        return report[report["missing_count"] > 0]

    def has_missing(self) -> bool:
        return bool(self.df.isna().any().any())

    def drop_missing(
        self,
        axis: int = 0,
        how: str = "any",
        subset: Optional[Sequence[str]] = None,
        thresh: Optional[int] = None,
    ) -> "CSVDataset":
        """Drop rows (axis=0) or columns (axis=1) containing NaNs.

        Mirrors ``DataFrame.dropna``. Returns self for chaining.
        """
        self.df = self.df.dropna(axis=axis, how=how, subset=subset, thresh=thresh)
        return self

    def fill_missing(
        self,
        strategy: str = "mean",
        columns: Optional[Iterable[str]] = None,
        fill_value=None,
    ) -> "CSVDataset":
        """Impute missing values.

        Parameters
        ----------
        strategy : {'mean', 'median', 'mode', 'constant', 'ffill', 'bfill'}
            - mean / median : numeric columns only.
            - mode          : works on any column; uses the most frequent value.
            - constant      : fill with ``fill_value``.
            - ffill / bfill : forward / backward fill.
        columns : iterable of str, optional
            Limit imputation to these columns. Defaults to all columns.
        fill_value :
            Required when strategy == 'constant'.
        """
        cols = list(columns) if columns is not None else self.df.columns.tolist()

        if strategy == "constant":
            if fill_value is None:
                raise ValueError("strategy='constant' needs a `fill_value`.")
            self.df[cols] = self.df[cols].fillna(fill_value)

        elif strategy in ("ffill", "bfill"):
            method = strategy  # 'ffill' or 'bfill'
            self.df[cols] = getattr(self.df[cols], method)()

        elif strategy in ("mean", "median"):
            for col in cols:
                if pd.api.types.is_numeric_dtype(self.df[col]):
                    value = getattr(self.df[col], strategy)()
                    self.df[col] = self.df[col].fillna(value)
            # non-numeric columns are silently skipped for mean/median

        elif strategy == "mode":
            for col in cols:
                mode = self.df[col].mode(dropna=True)
                if not mode.empty:
                    self.df[col] = self.df[col].fillna(mode.iloc[0])

        else:
            raise ValueError(f"Unknown strategy: {strategy!r}")

        return self

    def replace_nan(
            self,
            value=None,
            numeric_strategy: str = "median",
            columns: Optional[Iterable[str]] = None,
            report: bool = False,
        ) -> "CSVDataset":
        """Replace all NaN / missing values in one call.
 
        This is the "just clean everything" shortcut. Unlike ``fill_missing``,
        which applies a single strategy, this handles numeric and string
        columns together in a single pass.
 
        Two modes:
 
        1. Blanket replace -- pass a ``value`` and every NaN across the
           selected columns becomes that value.
               ds.replace_nan(0)
               ds.replace_nan("unknown")
 
        2. Type-aware replace (default, when ``value`` is None) --
           numeric columns are filled with their mean/median (set by
           ``numeric_strategy``), and string/categorical columns are filled
           with their most frequent value (mode).
               ds.replace_nan()                          # median + mode
               ds.replace_nan(numeric_strategy="mean")   # mean + mode
 
        Parameters
        ----------
        value :
            If given, use this single value for every NaN (mode 1).
        numeric_strategy : {'mean', 'median'}
            How to fill numeric columns in type-aware mode.
        columns : iterable of str, optional
            Limit the replacement to these columns. Defaults to all.
        report : bool
                If True, print how many NaNs were replaced per column.
     
            Returns ``self`` so it can be chained.
            """
        cols = list(columns) if columns is not None else self.df.columns.tolist()
        before = self.df[cols].isna().sum()
     
        if value is not None:
                # Mode 1: blanket replacement with one fixed value.
            self.df[cols] = self.df[cols].fillna(value)
        else:
                # Mode 2: type-aware replacement.
            if numeric_strategy not in ("mean", "median"):
                raise ValueError("numeric_strategy must be 'mean' or 'median'.")
            for col in cols:
                if self.df[col].isna().sum() == 0:
                    continue
                if pd.api.types.is_numeric_dtype(self.df[col]):
                    fill = getattr(self.df[col], numeric_strategy)()
                else:
                    mode = self.df[col].mode(dropna=True)
                    fill = mode.iloc[0] if not mode.empty else value
                self.df[col] = self.df[col].fillna(fill)
     
        if report:
            replaced = before[before > 0]
            if replaced.empty:
                print("replace_nan: nothing to replace.")
            else:
                print("replace_nan: values replaced per column")
            for col, n in replaced.items():
                print(f"  {col}: {int(n)}")
     
            return self

    # ------------------------------------------------------------------ #
    # CATEGORICAL VARIABLE HANDLING
    # ------------------------------------------------------------------ #
    def category_summary(self) -> dict[str, pd.Series]:
        """Value counts for each categorical column (handy for a quick look)."""
        return {
            col: self.df[col].value_counts(dropna=False)
            for col in self.categorical_columns()
        }

    def encode_categorical(
        self,
        method: str = "onehot",
        columns: Optional[Iterable[str]] = None,
        drop_first: bool = False,
    ) -> "CSVDataset":
        """Turn string/categorical columns into numbers.

        Parameters
        ----------
        method : {'onehot', 'label'}
            - onehot : one dummy column per category (pandas.get_dummies).
            - label  : map each category to an integer code.
        columns : iterable of str, optional
            Which columns to encode. Defaults to all detected categoricals.
        drop_first : bool
            For one-hot, drop the first level to avoid the dummy trap.

        Note: for 'label' encoding, the mapping learned per column is stored
        in ``self.label_maps`` so you can reverse it later.
        """
        cols = list(columns) if columns is not None else self.categorical_columns()

        if not cols:
            return self  # nothing to do

        if method == "onehot":
            self.df = pd.get_dummies(
                self.df, columns=cols, drop_first=drop_first
            )

        elif method == "label":
            if not hasattr(self, "label_maps"):
                self.label_maps: dict[str, dict] = {}
            for col in cols:
                codes, uniques = pd.factorize(self.df[col])
                # factorize marks NaN as -1; keep it as NaN instead
                codes = pd.Series(codes, index=self.df.index).replace(-1, np.nan)
                self.df[col] = codes
                self.label_maps[col] = dict(enumerate(uniques))

        else:
            raise ValueError(f"Unknown method: {method!r}")

        return self

    # ------------------------------------------------------------------ #
    # EDA CONVENIENCE
    # ------------------------------------------------------------------ #
    def summary(self) -> dict:
        """A one-glance overview dictionary."""
        return {
            "shape": self.df.shape,
            "columns": self.df.columns.tolist(),
            "dtypes": self.df.dtypes.astype(str).to_dict(),
            "numeric_columns": self.numeric_columns(),
            "categorical_columns": self.categorical_columns(),
            "total_missing": int(self.df.isna().sum().sum()),
            "duplicate_rows": int(self.df.duplicated().sum()),
            "memory_kb": round(self.df.memory_usage(deep=True).sum() / 1024, 1),
        }

    def describe(self, include_all: bool = True) -> pd.DataFrame:
        """Descriptive stats. ``include_all`` also covers categorical cols."""
        return self.df.describe(include="all" if include_all else None)

    def correlations(self, method: str = "pearson") -> pd.DataFrame:
        """Correlation matrix across numeric columns."""
        return self.df[self.numeric_columns()].corr(method=method)

    def outlier_flags(self, column: str, k: float = 1.5) -> pd.Series:
        """Boolean mask of IQR-based outliers in a numeric column.

        A value is an outlier if it falls outside
        [Q1 - k*IQR, Q3 + k*IQR]. Default k=1.5 is the usual Tukey rule.
        """
        if column not in self.numeric_columns():
            raise ValueError(f"{column!r} is not numeric.")
        q1, q3 = self.df[column].quantile([0.25, 0.75])
        iqr = q3 - q1
        low, high = q1 - k * iqr, q3 + k * iqr
        return (self.df[column] < low) | (self.df[column] > high)

In [4]:
df_train = CSVDataset("/kaggle/input/competitions/playground-series-s6e9/train.csv", na_values=["", "NA", "?"])

In [5]:
df_train

<CSVDataset rows=668665 cols=15 source='/kaggle/input/competitions/playground-series-s6e9/train.csv'>

In [6]:
df_test = CSVDataset("/kaggle/input/competitions/playground-series-s6e9/test.csv", na_values=["", "NA", "?"])

In [7]:
df_train = df_train.df.drop('id',axis=1)
y_train  = df_train['Will_Buy_EV']

In [8]:
x_train = df_train.drop('Will_Buy_EV',axis=1)

In [9]:
x_train=pd.get_dummies(x_train,columns=['Gender','City_Type','Current_Car_Type','Range_Anxiety_Level','Home_Charging_Possible','Subsidy_Available'], dtype=int)


In [10]:
df_test.df

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
0,668665,61,67725.0,16.9,2,7,4,4.0,Male,Suburban,Sedan,Yes,No,Low
1,668666,42,152835.0,41.9,2,9,9,4.0,Male,Urban,SUV,No,No,Low
2,668667,68,86877.0,53.3,1,10,11,4.0,Female,Urban,Sedan,No,No,Low
3,668668,39,46794.0,34.1,2,4,8,4.0,Female,Suburban,Sedan,Yes,No,Low
4,668669,55,112172.0,57.2,1,6,4,1.0,Female,Suburban,Sedan,Yes,Yes,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286566,955231,63,75830.0,23.9,1,3,3,1.0,Male,Suburban,SUV,Yes,No,Low
286567,955232,27,79563.0,55.2,2,7,5,3.0,Male,Suburban,Sedan,Yes,Yes,Low
286568,955233,30,48183.0,42.6,3,7,7,4.0,Female,Suburban,Sedan,Yes,No,Low
286569,955234,32,107845.0,5.0,2,14,7,1.0,Female,Urban,SUV,No,No,Low


In [11]:
x_test=pd.get_dummies(df_test.df,columns=['Gender','City_Type','Current_Car_Type','Range_Anxiety_Level','Home_Charging_Possible','Subsidy_Available'], dtype=int)

In [12]:
x_test = x_test.drop('id',axis=1)

In [13]:
x_train

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender_Female,Gender_Male,Gender_Other,...,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck,Range_Anxiety_Level_High,Range_Anxiety_Level_Low,Range_Anxiety_Level_Medium,Home_Charging_Possible_No,Home_Charging_Possible_Yes,Subsidy_Available_No,Subsidy_Available_Yes
0,66,92887.0,23.4,2,3,7,1.0,0,1,0,...,0,1,0,0,1,0,0,1,1,0
1,38,30000.0,5.0,1,2,2,4.0,0,1,0,...,1,0,0,0,1,0,0,1,1,0
2,26,94389.0,36.8,1,8,15,5.0,1,0,0,...,0,1,0,0,1,0,1,0,0,1
3,66,73580.0,23.7,2,6,9,3.0,0,1,0,...,0,0,0,0,1,0,0,1,1,0
4,54,57898.0,50.8,1,2,3,3.0,0,1,0,...,0,0,0,0,1,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,0,1,0,...,0,1,0,0,0,1,1,0,0,1
668661,32,129990.0,5.0,2,5,4,1.0,1,0,0,...,1,0,0,0,1,0,0,1,0,1
668662,64,121791.0,24.2,1,5,6,1.0,1,0,0,...,0,0,0,0,1,0,0,1,0,1
668663,51,115923.0,43.7,2,0,0,1.0,1,0,0,...,1,0,0,0,1,0,0,1,0,1


In [14]:
x_test

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender_Female,Gender_Male,Gender_Other,...,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck,Range_Anxiety_Level_High,Range_Anxiety_Level_Low,Range_Anxiety_Level_Medium,Home_Charging_Possible_No,Home_Charging_Possible_Yes,Subsidy_Available_No,Subsidy_Available_Yes
0,61,67725.0,16.9,2,7,4,4.0,0,1,0,...,0,1,0,0,1,0,0,1,1,0
1,42,152835.0,41.9,2,9,9,4.0,0,1,0,...,1,0,0,0,1,0,1,0,1,0
2,68,86877.0,53.3,1,10,11,4.0,1,0,0,...,0,1,0,0,1,0,1,0,1,0
3,39,46794.0,34.1,2,4,8,4.0,1,0,0,...,0,1,0,0,1,0,0,1,1,0
4,55,112172.0,57.2,1,6,4,1.0,1,0,0,...,0,1,0,0,1,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286566,63,75830.0,23.9,1,3,3,1.0,0,1,0,...,1,0,0,0,1,0,0,1,1,0
286567,27,79563.0,55.2,2,7,5,3.0,0,1,0,...,0,1,0,0,1,0,0,1,0,1
286568,30,48183.0,42.6,3,7,7,4.0,1,0,0,...,0,1,0,0,1,0,0,1,1,0
286569,32,107845.0,5.0,2,14,7,1.0,1,0,0,...,1,0,0,0,1,0,1,0,1,0


In [15]:
y_train = y_train.map({'Yes': 1, 'No': 0})

In [16]:
y_train

0         0
1         0
2         1
3         0
4         0
         ..
668660    0
668661    0
668662    0
668663    0
668664    0
Name: Will_Buy_EV, Length: 668665, dtype: int64

In [17]:
from xgboost import XGBClassifier as xgbc
import lightgbm as lgb
    

# Train the model
model = xgbc(
    objective='binary:logistic',
    n_estimators=500,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.7,
    colsample_bytree=0.3,
    use_label_encoder=False,
    eval_metric='logloss',
    verbose=2
)

model.fit(x_train,y_train)

# 4. Predict using the DMatrix
predictions = model.predict(x_train)
y_pred = model.predict(x_test)

#predictions=(predictions >= 0.5).astype(int)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [03:08:54] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder", "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [18]:
submission_df = pd.DataFrame({'id': df_test.df['id'], 'Will_Buy_EV': y_pred})
submission_df.to_csv("submission.csv", index=False)